In [ ]:
!rm -rf /kaggle/working/activeqa_final
!git clone https://github.com/Suhas-Koheda/activeqa_final.git /kaggle/working/activeqa_final
import os
os.environ['HF_TOKEN'] = 'PASTE_YOUR_TOKEN_HERE'

In [ ]:
!cd /kaggle/working/activeqa_final && python scripts/kaggle_run_all.py --from-base

In [ ]:
import os
token = os.environ.get("HF_TOKEN")
if not token:
    raise RuntimeError("set os.environ['HF_TOKEN'] in the first cell, e.g. os.environ['HF_TOKEN'] = 'hf_...'")
import shutil
import sys
from pathlib import Path
ROOT = Path("/kaggle/working/activeqa_final")
sys.path.insert(0, str(ROOT / "src"))
BASE = ROOT / "models" / "MultiIndicParaphraseGenerationSS"
ADAPTER = ROOT / "models" / "indicbartss-reformulator-lora-rl" / "adapter"
OUT = ROOT / "models" / "hf-push"
if not ADAPTER.is_dir():
    raise RuntimeError(f"RL adapter missing: {ADAPTER} - run the training cell first")
from huggingface_hub import HfApi, login
login(token=token)
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
model = AutoModelForSeq2SeqLM.from_pretrained(str(BASE))
model = PeftModel.from_pretrained(model, str(ADAPTER))
merged = model.merge_and_unload()
tok = AutoTokenizer.from_pretrained(str(BASE), use_fast=False, do_lower_case=False, keep_accents=True)
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
merged.save_pretrained(str(OUT))
tok.save_pretrained(str(OUT))
card_lines = [
    "---",
    "language: te",
    "license: mit",
    "tags:",
    "- question-reformulation",
    "- telugu",
    "- reinforcement-learning",
    "- lora",
    "- activeqa",
    "---",
    "",
    "# ActiveQATeluguReformulatorRL (merged LoRA)",
    "",
    "MultiIndicParaphraseGenerationSS post-trained with a validation-gated RL loop",
    "(entity-preservation reward, QA reward l3cube-pune/telugu-question-answering-squad-bert).",
    "No SFT: the LoRA trained directly from the raw checkpoint on 804 Telugu questions,",
    "best checkpoint selected on 200 held-out dev questions.",
    "",
    "Dataset: sampled from ai4bharat/Indic-Rag-Suite (config te, LLM-generated QA from",
    "Telugu Wikipedia). Answers are silver and not human-verified. Research artifact;",
    "not a production QA system.",
]
(OUT / "README.md").write_text("\n".join(card_lines) + "\n")
api = HfApi(token=token)
api.create_repo(repo_id="nlpctx/activeqa-telugu-reformulator-rl", exist_ok=True)
api.upload_folder(repo_id="nlpctx/activeqa-telugu-reformulator-rl", folder_path=str(OUT), repo_type="model")
print("pushed https://huggingface.co/nlpctx/activeqa-telugu-reformulator-rl")

In [ ]:
import json
import random
import sys
from pathlib import Path
ROOT = Path("/kaggle/working/activeqa_final")
sys.path.insert(0, str(ROOT / "src"))
import torch
from activeqa.reformulator import Reformulator
from activeqa.qa_model import TeluguQA
from activeqa.preprocessing import normalize
from activeqa.rewards import token_f1
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); random.seed(42)
ref = Reformulator("nlpctx/activeqa-telugu-reformulator-rl",
                   cache_dir="/kaggle/working/hf_demo_cache", device=device)
qa = TeluguQA("l3cube-pune/telugu-question-answering-squad-bert",
              cache_dir=str(ROOT / "models"), device=device)
rows = [json.loads(l) for l in open(ROOT / "data/indic_rag_te/dev.jsonl", encoding="utf-8")][:3]
for row in rows:
    q = normalize(row["question"])
    ctx = row["context"]
    gold = row["answers"]
    ans_q, _, score_q = qa.answer_with_score(q, ctx)
    acts = ref.sample_actions(q, n=6, max_input=96, max_new=64, do_sample=True)
    valid = [a for a in acts if a["valid"]]
    print("=" * 78)
    print("Q :", q)
    print("   gold:", gold)
    print(f"   ANSWER(original) [{score_q:.2f}]: {ans_q}   f1={token_f1(ans_q, gold):.2f}")
    if not valid:
        print("   (all candidates invalid)")
    for i, a in enumerate(valid, 1):
        ans, _, s = qa.answer_with_score(a["candidate"], ctx)
        print(f"   cand {i} (valid): {a['candidate']}")
        print(f"      ANSWER [{s:.2f}]: {ans}   f1={token_f1(ans, gold):.2f}")
print("=" * 78)
print("valid-rate on last question:", sum(a["valid"] for a in acts), "/", len(acts))